 Grain Boundary Chemistry — Synthetic Data & Analysis
 Inspired by Zhou et al., "Revealing in-plane grain boundarycomposition features through machine learning from atom probe
# %  tomography data."

In [ ]:
!pip install numpy matplotlib scikit-learn jupyter

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Create output directory for figures
Path("figures").mkdir(exist_ok=True)

# Reproducible random data
rng = np.random.default_rng(42)

# 1. Generate artificial atom coordinates
n_atoms = 30_000
x = rng.uniform(-5, 5, n_atoms)
y = rng.uniform(-5, 5, n_atoms)

# 2. Define a curved grain boundary
def true_boundary(y):
    return 0.7 * np.sin(0.7 * y)

boundary_x = true_boundary(y)
distance = np.abs(x - boundary_x)

# 3. Make solute concentration higher near the boundary
# Concentration also varies along the boundary.
along_boundary = (
    0.5 + 0.5 * np.sin(1.8 * y)
)

probability = (
    0.03
    + 0.55 * np.exp(-distance / 0.30)
    * (0.25 + 0.75 * along_boundary)
)

probability = np.clip(probability, 0, 1)

# True means solute atom; False means host atom
is_solute = rng.random(n_atoms) < probability

print("Number of atoms:", n_atoms)
print("Solute fraction:", round(is_solute.mean(), 3))


In [ ]:

plt.figure(figsize=(9, 6))

plt.scatter(
    x[~is_solute], y[~is_solute],
    s=3, alpha=0.45, label="Host atoms"
)
plt.scatter(
    x[is_solute], y[is_solute],
    s=7, alpha=0.8, label="Solute atoms"
)

y_line = np.linspace(-5, 5, 300)
plt.plot(
    true_boundary(y_line), y_line,
    "k--", linewidth=2, label="True boundary"
)

plt.xlabel("x position")
plt.ylabel("y position")
plt.title("Synthetic Atom Map")
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig("figures/01_atom_map.png", dpi=200)
plt.show()


In [ ]:

# Divide space into rectangular bins
x_edges = np.linspace(-5, 5, 41)
y_edges = np.linspace(-5, 5, 21)

# Count all atoms and solute atoms in each cell
all_counts, _, _ = np.histogram2d(
    x, y, bins=[x_edges, y_edges]
)

solute_counts, _, _ = np.histogram2d(
    x[is_solute], y[is_solute],
    bins=[x_edges, y_edges]
)

# Solute fraction per cell
fraction = np.divide(
    solute_counts,
    all_counts,
    out=np.full_like(all_counts, np.nan),
    where=all_counts > 0
)

x_centers = (x_edges[:-1] + x_edges[1:]) / 2
y_centers = (y_edges[:-1] + y_edges[1:]) / 2

# For each y band, find the x bin with the
# highest measured solute fraction.
estimated_x = []

for j in range(len(y_centers)):
    profile = fraction[:, j]
    valid = np.isfinite(profile)

    if valid.any():
        scores = np.where(valid, profile, -np.inf)
        best_bin = np.argmax(scores)
        estimated_x.append(x_centers[best_bin])
    else:
        estimated_x.append(np.nan)

estimated_x = np.array(estimated_x)


In [ ]:

plt.figure(figsize=(9, 6))

plt.scatter(
    x, y, c=is_solute,
    cmap="coolwarm", s=2, alpha=0.18
)

plt.plot(
    true_boundary(y_line), y_line,
    "k--", linewidth=2, label="True boundary"
)

plt.plot(
    estimated_x, y_centers,
    "go-", markersize=4, linewidth=1.5,
    label="Estimated boundary"
)

plt.xlabel("x position")
plt.ylabel("y position")
plt.title("Boundary Detection from Solute Concentration")
plt.legend()
plt.tight_layout()
plt.savefig("figures/02_boundary_detection.png", dpi=200)
plt.show()


In [ ]:

# Assign each atom to its nearest estimated boundary position.
# Interpolate the estimated boundary between y-band centers.
valid = np.isfinite(estimated_x)

boundary_at_atoms = np.interp(
    y,
    y_centers[valid],
    estimated_x[valid]
)

distance_estimated = np.abs(x - boundary_at_atoms)

# Select atoms close to the estimated boundary
near_boundary = distance_estimated < 0.45

# Divide the boundary into segments
segment_edges = np.linspace(-5, 5, 16)
segment_centers = (
    segment_edges[:-1] + segment_edges[1:]
) / 2

local_fraction = []
local_counts = []

for i in range(len(segment_edges) - 1):
    in_segment = (
        (y >= segment_edges[i])
        & (y < segment_edges[i + 1])
        & near_boundary
    )

    count = in_segment.sum()
    local_counts.append(count)

    if count > 0:
        local_fraction.append(is_solute[in_segment].mean())
    else:
        local_fraction.append(np.nan)

local_fraction = np.array(local_fraction)

# Plot composition variation along the boundary
plt.figure(figsize=(9, 5))
plt.plot(
    segment_centers, local_fraction,
    "o-", label="Near-boundary solute fraction"
)
plt.axhline(
    is_solute.mean(), color="gray", linestyle="--",
    label="Overall solute fraction"
)

plt.xlabel("Position along boundary (y)")
plt.ylabel("Solute fraction")
plt.title("In-Plane Chemical Variation")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("figures/03_in_plane_chemistry.png", dpi=200)
plt.show()


In [ ]:

# Compare estimated and true boundary positions
true_at_centers = true_boundary(y_centers)

valid = np.isfinite(estimated_x)

errors = estimated_x[valid] - true_at_centers[valid]

mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))

print(f"Boundary MAE:  {mae:.3f} coordinate units")
print(f"Boundary RMSE: {rmse:.3f} coordinate units")

plt.figure(figsize=(8, 4))
plt.plot(
    y_centers[valid], errors,
    "o-"
)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Position along boundary (y)")
plt.ylabel("Estimated - true x position")
plt.title("Boundary Localization Error")
plt.tight_layout()
plt.savefig("figures/04_boundary_error.png", dpi=200)
plt.show()


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Label atoms by distance from the known boundary
is_boundary = distance < 0.35

# Input features: position and chemical identity
X = np.column_stack([x, y, is_solute.astype(int)])
target = is_boundary.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, target,
    test_size=0.25,
    random_state=42,
    stratify=target
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    target_names=["Interior", "Near boundary"]
))
